# Comparative Analysis of XAI Methods for Bone Fracture Detection

This notebook compares Grad-CAM, LIME, and SHAP for explaining ResNet-50 predictions on X-ray images.

In [1]:
import torch
import numpy as np
from src.model import load_model, get_transforms
from src.explain import Explainability
from src.utils import load_image, preprocess_image
import matplotlib.pyplot as plt
import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries

ModuleNotFoundError: No module named 'src'

In [ ]:
# Load model and sample image
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_model()
model.to(device)

# Replace with actual image path
image_path = 'data/sample_xray.png'
image = load_image(image_path)
transforms = get_transforms()
input_tensor = preprocess_image(image, transforms)

explainer = Explainability(model, device)

In [ ]:
# Grad-CAM
grad_cam_attr = explainer.grad_cam(input_tensor, 1)  # Assuming fracture class is 1
grad_cam_heatmap = explainer.generate_heatmap(grad_cam_attr, np.array(image.resize((224, 224))))

plt.figure(figsize=(6, 6))
plt.imshow(grad_cam_heatmap)
plt.title('Grad-CAM Heatmap')
plt.axis('off')
plt.show()

In [ ]:
# LIME
def predict_fn(images):
    images = torch.stack([transforms(Image.fromarray(img.astype('uint8'))) for img in images])
    images = images.to(device)
    with torch.no_grad():
        outputs = model(images)
        return torch.nn.functional.softmax(outputs, dim=1).cpu().numpy()

lime_explainer = lime_image.LimeImageExplainer()
lime_explanation = explainer.lime_explanation(np.array(image), predict_fn)

temp, mask = lime_explanation.get_image_and_mask(lime_explanation.top_labels[0], positive_only=True, num_features=5, hide_rest=False)
plt.figure(figsize=(6, 6))
plt.imshow(mark_boundaries(temp / 255.0, mask))
plt.title('LIME Explanation')
plt.axis('off')
plt.show()

In [ ]:
# SHAP
# Create background dataset (e.g., random images or subset of training data)
# For demonstration, using a few perturbed versions
background = torch.randn(10, 3, 224, 224).to(device)
shap_values = explainer.shap_explanation(background, input_tensor)

# Visualize SHAP values
shap.image_plot(shap_values, input_tensor.cpu().numpy())

## Comparative Analysis

- **Grad-CAM**: Provides class-specific heatmaps by focusing on the last convolutional layer. Good for highlighting regions contributing to the prediction.
- **LIME**: Generates explanations by approximating the model locally with an interpretable model. Shows superpixels that influence the prediction.
- **SHAP**: Provides global feature importance and can handle complex interactions. More computationally intensive.

Compare the explanations visually and quantitatively (e.g., using metrics like faithfulness or stability).